# Exploration interactive — divergences textuelles ANS ↔ OFS

Source canonique : `reports/note_merges.csv`.

Version notebook du fichier `scripts/explore/2026-05-17_divergences_textuelles.py`. Régénérée via `_convert_to_ipynb.py`. **Ne pas éditer directement le notebook** — modifier le `.py` et re-convertir.

## Test

In [1]:
print("hello")
1 + 1

hello


2

## Init : chargement du contexte recode-icd

In [2]:
from __future__ import annotations

import random
import re
from textwrap import shorten

import polars as pl

from recode_icd.utils.loaders_dev import load_exploration_context

ctx = load_exploration_context()
note_merges = ctx.reports["note_merges"]

# Sous-ensemble d'intérêt : seulement les VRAIES divergences (texte ANS
# encore différent de OFS après normalisation complète).
divergences = note_merges.filter(pl.col("difference_significative"))

print(f"note_merges totales : {len(note_merges):,}")
print(f"  - identiques après normalisation : "
      f"{note_merges['libelles_identiques_apres_normalisation'].sum():,}")
print(f"  - vraies divergences             : {len(divergences):,}")

note_merges totales : 5,603
  - identiques après normalisation : 178
  - vraies divergences             : 5,425


In [8]:
master  = ctx.ofs["master"]
libelle = ctx.ofs["libelle"]
exclude = ctx.ofs["exclude"]

# Libellé systématique (source='S') par SID — pour le code source ET le redirect
sid_to_label = (
    libelle.filter(pl.col("source") == "S")
           .select("SID", pl.col("libelle").alias("label"))
           .unique(subset="SID", keep="first")
)

# Texte de l'exclusion lui-même (source='E') par LID
lid_to_excl_text = (
    libelle.filter(pl.col("source") == "E")
           .select("LID", pl.col("libelle").alias("exclusion_text"))
)

# Code/abbrev/type par SID — réutilisé pour la source et le redirect
sid_to_code = master.select(
    "SID", "code", "abbrev",
    pl.col("type").alias("ofs_type"),
    pl.col("valid").alias("source_valid"),
)

exclusions_ofs = (
    exclude
      # SID = code qui porte l'exclusion
      .join(sid_to_code,     on="SID", how="inner")
      .join(sid_to_label,    on="SID", how="left")
      # LID = texte de l'exclusion
      .join(lid_to_excl_text, on="LID", how="left")
      # excl = code de redirection (0 = pas de cible, NULL après join)
      .join(
          sid_to_code.rename({
              "SID": "excl",
              "code": "redirect_code",
              "abbrev": "redirect_abbrev",
              "ofs_type": "redirect_type",
              "source_valid": "redirect_valid",
          }),
          on="excl", how="left",
      )
      .join(
          sid_to_label.rename({"SID": "excl", "label": "redirect_label"}),
          on="excl", how="left",
      )
      .select(
          "code", "label", "ofs_type",
          "exclusion_text",
          "redirect_code", "redirect_label", "redirect_type",
          "daget", "plus",
          # IDs et flags pour debug
          "SID", "LID", "excl", "source_valid", "redirect_valid",
      )
)

In [9]:
ans = ctx.ans

# Pré-calc : URIs structurées → codes (suffixe après le dernier "/")
structured_redirects = ans.select(
    "code",
    pl.col("structured_exclusions").list.eval(
        pl.element().str.split("/").list.last()
    ).alias("structured_redirects"),
)

# Une ligne par (code, exclusion_text) — explode de exclusion_notes
exclusions_ans = (
    ans
      .join(structured_redirects, on="code", how="left")
      .filter(pl.col("exclusion_notes").list.len() > 0)
      .select(
          "code",
          pl.col("label").alias("code_label"),
          "type",
          pl.col("exclusion_notes").alias("exclusion_text"),
          # list[str] des codes cibles — au niveau code, PAS du texte spécifique
          "structured_redirects",
      )
      .explode("exclusion_text")
)

## Stats globales sur les vraies divergences

In [12]:
print("Par type de note :")
print(divergences.group_by("type").len().sort("len", descending=True))

print("\nNombre de codes uniques touchés :", divergences["code"].n_unique())

# Distribution simple longueur OFS vs longueur ANS
with_lens = divergences.with_columns(
    pl.col("texte_retenu").str.len_chars().alias("len_ofs"),
    pl.col("texte_alternatif_ans").str.len_chars().alias("len_ans"),
).with_columns(
    (pl.col("len_ans") - pl.col("len_ofs")).alias("delta"),
)

print("\nSens du delta (ANS - OFS) :")
print(
    with_lens.select(
        (pl.col("delta") > 0).sum().alias("ans_plus_long"),
        (pl.col("delta") < 0).sum().alias("ofs_plus_long"),
        (pl.col("delta") == 0).sum().alias("egales"),
    )
)

Par type de note :
shape: (2, 2)
┌───────────┬──────┐
│ type      ┆ len  │
│ ---       ┆ ---  │
│ str       ┆ u32  │
╞═══════════╪══════╡
│ exclusion ┆ 4372 │
│ inclusion ┆ 1053 │
└───────────┴──────┘

Nombre de codes uniques touchés : 2193

Sens du delta (ANS - OFS) :
shape: (1, 3)
┌───────────────┬───────────────┬────────┐
│ ans_plus_long ┆ ofs_plus_long ┆ egales │
│ ---           ┆ ---           ┆ ---    │
│ u32           ┆ u32           ┆ u32    │
╞═══════════════╪═══════════════╪════════╡
│ 5409          ┆ 0             ┆ 16     │
└───────────────┴───────────────┴────────┘


## Helper d'affichage d'une divergence donnée (code, type)

In [13]:
def show_divergence(code: str, note_type: str | None = None) -> None:
    """Affiche côté à côte les versions OFS et ANS d'une divergence."""
    q = divergences.filter(pl.col("code") == code)
    if note_type is not None:
        q = q.filter(pl.col("type") == note_type)

    if q.is_empty():
        print(f"Aucune divergence pour code={code!r} type={note_type!r}.")
        return

    for row in q.iter_rows(named=True):
        print(f"━━━ {row['code']} ({row['type']}) ━━━")
        print(f"  OFS ({len(row['texte_retenu'])} car):")
        print(f"    {row['texte_retenu']}")
        print(f"  ANS ({len(row['texte_alternatif_ans'])} car):")
        print(f"    {row['texte_alternatif_ans']}")
        print()


# Quelques codes témoins pour démo : adapter à ce qu'on veut creuser.
show_divergence("F02.00")
show_divergence("A02.1")
show_divergence("J45.0", "exclusion")

Aucune divergence pour code='F02.00' type=None.
Aucune divergence pour code='A02.1' type=None.
Aucune divergence pour code='J45.0' type='exclusion'.


## Échantillonnage stratifié — 50 cas

In [14]:
SEED = 42
rng = random.Random(SEED)


def _sample(df: pl.DataFrame, n: int) -> list[dict[str, object]]:
    rows = df.to_dicts()
    rng.shuffle(rows)
    return rows[:n]


def _pretty_print_sample(label: str, rows: list[dict[str, object]]) -> None:
    print(f"\n━━━ {label} ({len(rows)} cas) ━━━")
    for r in rows:
        ofs = str(r["texte_retenu"])
        ans = str(r["texte_alternatif_ans"])
        print(f"  {r['code']} ({r['type']}) — OFS={len(ofs)} / ANS={len(ans)}")
        print(f"    OFS : {shorten(ofs, width=120, placeholder='…')}")
        print(f"    ANS : {shorten(ans, width=120, placeholder='…')}")


# Stratifications
_lens = with_lens

samples_ans_plus_long = _sample(
    _lens.filter(pl.col("delta") > 5),
    10,
)
samples_ofs_plus_long = _sample(
    _lens.filter(pl.col("delta") < -5),
    10,
)
samples_similar = _sample(
    _lens.filter(pl.col("delta").abs() <= 5),
    10,
)

# Chapitres cliniques très annotés : I/J (cardio-respiratoire) et S/T (lésions) et C/D (tumeurs)
samples_clinical = _sample(
    _lens.filter(
        pl.col("code").str.starts_with("C")
        | pl.col("code").str.starts_with("I")
        | pl.col("code").str.starts_with("S")
    ),
    10,
)

samples_random = _sample(_lens, 10)

_pretty_print_sample("ANS strictement plus long", samples_ans_plus_long)
_pretty_print_sample("OFS strictement plus long", samples_ofs_plus_long)
_pretty_print_sample("Longueurs similaires (|Δ| ≤ 5)", samples_similar)
_pretty_print_sample("Chapitres cliniques (C/I/S)", samples_clinical)
_pretty_print_sample("Random (contrôle)", samples_random)


━━━ ANS strictement plus long (10 cas) ━━━
  Q82.5 (exclusion) — OFS=15 / ANS=165
    OFS : naevus (à) SAI
    ANS : lentigo [L81.4] nævus (à) : - SAI [D22.-] - arachnéen [I78.1] - mélanocytes [D22.-] - pigmentaire [D22.-] - stellaire…
  E22.0 (exclusion) — OFS=85 / ANS=167
    OFS : hypersécrétion du pancréas endocrine du "releasing factor" de l'hormone de croissance
    ANS : gigantisme constitutionnel [E34.4] haute stature constitutionnelle [E34.4] hypersécrétion du pancréas endocrine du…
  Z92.4 (exclusion) — OFS=46 / ANS=140
    OFS : présence d'implants et de greffes fonctionnels
    ANS : états postchirurgicaux [Z98.-] greffe d'organe ou de tissu [Z94.-] présence d'implants et de greffes fonctionnels…
  R52 (exclusion) — OFS=19 / ANS=476
    OFS : douleur (de) épaule
    ANS : céphalée [R51] colique néphrétique [N23] douleur (de) : - abdominale [R10.-] - articulaire [M25.5] - dent [K08.8] - dos…
  W27 (inclusion) — OFS=11 / ANS=178
    OFS : scie à main
    ANS : aiguille bêche

In [ ]:
# === Cellule "Affichage des divergences code  E22.0" ===
code = "E22.0"
print("ANS")
print(exclusions_ans.filter(pl.col("code") == code))
print("OFS")
print(exclusions_ofs.filter(pl.col("code") == code))

ANS
shape: (1, 5)
┌───────┬────────────────────────────┬──────────┬────────────────────────┬──────────────────────┐
│ code  ┆ code_label                 ┆ type     ┆ exclusion_text         ┆ structured_redirects │
│ ---   ┆ ---                        ┆ ---      ┆ ---                    ┆ ---                  │
│ str   ┆ str                        ┆ str      ┆ str                    ┆ list[str]            │
╞═══════╪════════════════════════════╪══════════╪════════════════════════╪══════════════════════╡
│ E22.0 ┆ Acromégalie et gigantisme  ┆ category ┆ gigantisme             ┆ ["E16.8", "E34.4"]   │
│       ┆ hypo…                      ┆          ┆ constitutionnel [E3…   ┆                      │
└───────┴────────────────────────────┴──────────┴────────────────────────┴──────────────────────┘
OFS
shape: (3, 14)
┌───────┬───────────────┬──────────┬──────────────┬───┬───────┬──────┬──────────────┬──────────────┐
│ code  ┆ label         ┆ ofs_type ┆ exclusion_te ┆ … ┆ LID   ┆ excl ┆ source_

## Détection de patterns automatiques

In [ ]:
CODE_REF_RE = re.compile(r"\[[A-Z]\d{2}(?:\.\d+)?\]")

with_patterns = divergences.with_columns(
    pl.col("texte_retenu").map_elements(
        lambda t: bool(CODE_REF_RE.search(t or "")),
        return_dtype=pl.Boolean,
    ).alias("ofs_has_code_ref"),
    pl.col("texte_alternatif_ans").map_elements(
        lambda t: bool(CODE_REF_RE.search(t or "")),
        return_dtype=pl.Boolean,
    ).alias("ans_has_code_ref"),
    pl.struct(["texte_retenu", "texte_alternatif_ans"]).map_elements(
        lambda s: (
            (s["texte_retenu"] or "") in (s["texte_alternatif_ans"] or "")
            and len(s["texte_retenu"] or "") < len(s["texte_alternatif_ans"] or "")
            and len(s["texte_retenu"] or "") > 0
        ),
        return_dtype=pl.Boolean,
    ).alias("ofs_substring_of_ans"),
    pl.struct(["texte_retenu", "texte_alternatif_ans"]).map_elements(
        lambda s: (
            (s["texte_alternatif_ans"] or "") in (s["texte_retenu"] or "")
            and len(s["texte_alternatif_ans"] or "") < len(s["texte_retenu"] or "")
            and len(s["texte_alternatif_ans"] or "") > 0
        ),
        return_dtype=pl.Boolean,
    ).alias("ans_substring_of_ofs"),
)

print("\n=== Patterns détectés ===")
print(
    with_patterns.select(
        pl.col("ofs_has_code_ref").sum().alias("ofs_avec_code_ref"),
        pl.col("ans_has_code_ref").sum().alias("ans_avec_code_ref"),
        pl.col("ofs_substring_of_ans").sum().alias("ofs_substring_ans"),
        pl.col("ans_substring_of_ofs").sum().alias("ans_substring_ofs"),
    )
)

## Creuser les cas avec OFS strictement plus long (rares mais intéressants)

In [ ]:
# Hypothèse : OFS conserve parfois un qualificatif clinique perdu côté ANS.
ofs_longer_full = _lens.filter(pl.col("delta") < -5).sort("delta")
print(f"\n{len(ofs_longer_full)} cas où OFS > ANS de plus de 5 caractères :\n")
for r in ofs_longer_full.head(10).iter_rows(named=True):
    print(f"  {r['code']} ({r['type']})")
    print(f"    OFS : {r['texte_retenu']}")
    print(f"    ANS : {r['texte_alternatif_ans']}")
    print()

## creuser un chapitre / un code particulier

In [ ]:
# === Cellule "Affichage des divergences code  I78.1" ===
code = "I78.1"
print("ANS")
print(exclusions_ans.filter(pl.col("code") == code))
print("OFS")
print(exclusions_ofs.filter(pl.col("code") == code))

ANS
shape: (1, 5)
┌───────┬────────────────────────┬──────────┬──────────────────┬──────────────────────┐
│ code  ┆ code_label             ┆ type     ┆ exclusion_text   ┆ structured_redirects │
│ ---   ┆ ---                    ┆ ---      ┆ ---              ┆ ---                  │
│ str   ┆ str                    ┆ str      ┆ str              ┆ list[str]            │
╞═══════╪════════════════════════╪══════════╪══════════════════╪══════════════════════╡
│ I78.1 ┆ Nævus, non néoplasique ┆ category ┆ nævus (à) (en) : ┆ ["Q82.5", "D22"]     │
│       ┆                        ┆          ┆  - SAI [D22.-…   ┆                      │
└───────┴────────────────────────┴──────────┴──────────────────┴──────────────────────┘
OFS
shape: (11, 14)
┌───────┬───────────────┬──────────┬──────────────┬───┬───────┬──────┬──────────────┬──────────────┐
│ code  ┆ label         ┆ ofs_type ┆ exclusion_te ┆ … ┆ LID   ┆ excl ┆ source_valid ┆ redirect_val │
│ ---   ┆ ---           ┆ ---      ┆ xt           ┆   ┆ 

# === Test de vérité I78.1 ===

In [18]:
# === Test de vérité I78.1 ===
# On va voir CE QUE CONTIENNENT VRAIMENT les deux sources, sans
# passer par aucune logique de merge ni de jointure complexe.

print("=" * 70)
print("CÔTÉ OFS — Table EXCLUDE jointe à LIBELLE pour I78.1")
print("=" * 70)

# 1. Trouver le SID d'I78.1 dans MASTER
sid_i781 = (
    ctx.ofs["master"]
    .filter(pl.col("code") == "I78.1")
    .select("SID")
    .item()
)
print(f"SID de I78.1 = {sid_i781}\n")

# 2. Récupérer toutes les exclusions de ce SID
exclusions_brutes_ofs = (
    ctx.ofs["exclude"]
    .filter(pl.col("SID") == sid_i781)
    .join(
        ctx.ofs["libelle"].select(["LID", "FR_OMS"]),
        on="LID",
        how="left"
    )
)
print(f"Nombre de lignes d'exclusion OFS pour I78.1 : "
      f"{len(exclusions_brutes_ofs)}")
print()
for row in exclusions_brutes_ofs.iter_rows(named=True):
    print(f"  LID={row['LID']}, texte=\"{row['FR_OMS']}\"")
    print()

print()
print("=" * 70)
print("CÔTÉ ANS — Parquet OWL pour I78.1")
print("=" * 70)

ans_i781 = ctx.ans.filter(pl.col("code") == "I78.1")
exclusion_notes_ans = ans_i781.select("exclusion_notes").item()
print(f"Nombre d'exclusion_notes ANS pour I78.1 : "
      f"{len(exclusion_notes_ans)}")
print()
for note in exclusion_notes_ans:
    print(f"  texte=\"{note}\"")
    print()

CÔTÉ OFS — Table EXCLUDE jointe à LIBELLE pour I78.1
SID de I78.1 = 4138

Nombre de lignes d'exclusion OFS pour I78.1 : 11

  LID=29901, texte="naevus (à) (en) bleu"

  LID=29904, texte="naevus (à) (en) mélanocytes"

  LID=29905, texte="naevus (à) (en) pigmentaire"

  LID=29906, texte="naevus (à) (en) pileux"

  LID=29907, texte="naevus (à) (en) SAI"

  LID=29902, texte="naevus (à) (en) flammeus"

  LID=29903, texte="naevus (à) (en) fraise"

  LID=29908, texte="naevus (à) (en) sanguin"

  LID=29909, texte="naevus (à) (en) tache de vin"

  LID=29910, texte="naevus (à) (en) vasculaire SAI"

  LID=29911, texte="naevus (à) (en) verruqueux"


CÔTÉ ANS — Parquet OWL pour I78.1
Nombre d'exclusion_notes ANS pour I78.1 : 1

  texte="nævus (à) (en) :
 - SAI [D22.-] 
 - bleu [D22.-] 
 - flammeus [Q82.5] 
 - fraise [Q82.5] 
 - mélanocytes [D22.-] 
 - pigmentaire [D22.-] 
 - pileux [D22.-] 
 - sanguin [Q82.5] 
 - tache de vin [Q82.5] 
 - vasculaire SAI [Q82.5] 
 - verruqueux [Q82.5] 
"



## (vide) — exporter une sélection en CSV pour partage manuel